In [ ]:
# ─────────────────────────────────────────
# Supervised Fine-Tuning (SFT) — Stage 2
# Loads the pre-trained model and trains it
# on curated instruction-style data
# ─────────────────────────────────────────
import json
import torch
import torch.nn as nn
from torch.nn import functional as F
from model import BigramLanguageModel

In [ ]:
# ─────────────────────────────────────────
# 10. Load tokenizer
# This is exactly what happens when you call
# AutoTokenizer.from_pretrained(...) in HuggingFace
# ─────────────────────────────────────────
with open("trained-model/tokenizer.json", "r", encoding="utf-8") as f:
    tokenizer = json.load(f)

stoi = tokenizer["stoi"]
itos = {int(k): v for k, v in tokenizer["itos"].items()}  # JSON keys are strings, convert back to int

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [ ]:
# ─────────────────────────────────────────
# 11. Load config
# This shows the details of model architecture
# ─────────────────────────────────────────
with open("trained-model/config.json", "r") as f:
    config = json.load(f)

vocab_size = config["vocab_size"]
print(f"Model type:  {config['model_type']}")
print(f"Vocab size:  {vocab_size}")

In [ ]:
# ─────────────────────────────────────────
# 12. Load the saved model
# This simulates loading a pre-trained checkpoint
# ─────────────────────────────────────────
from model import BigramLanguageModel

m = BigramLanguageModel(vocab_size)                                            # same architecture
m.load_state_dict(torch.load("trained-model/bigram_pretrained.pt"))           # load pre-trained weights as starting point
print("Pre-trained weights loaded successfully")

In [ ]:
# ─────────────────────────────────────────
# 13. SFT Dataset
# Real (instruction → response) pairs.
# This is exactly the format used by
# InstructGPT, Alpaca, and LLaMA-chat.
# Only 3-4 examples needed to demonstrate
# the idea — repeated to create enough tokens.
# ─────────────────────────────────────────

sft_pairs = [
    {
        "instruction": "What is gravity?",
        "response":    "Gravity is a force that pulls objects toward each other. The larger the object, the stronger its pull. It is why we stay on the ground and why planets orbit the sun."
    },
    {
        "instruction": "What is the capital of France?",
        "response":    "The capital of France is Paris. It is the largest city in France and has been its political and cultural centre for centuries."
    },
    {
        "instruction": "Explain what a neural network is.",
        "response":    "A neural network is a system of layers made of simple units called neurons. Each neuron takes numbers as input, multiplies them by learned weights, and passes the result forward. By training on many examples, the network learns weights that map inputs to correct outputs."
    },
    {
        "instruction": "What causes the seasons?",
        "response":    "Seasons are caused by the tilt of the Earth as it orbits the sun. When the northern hemisphere tilts toward the sun it receives more direct light and heat, giving summer. When it tilts away, it receives less light, giving winter."
    },
]


def format_sft(pair):
    """
    Wrap each pair in a prompt template.
    This is the Alpaca / LLaMA-chat format.
    The model learns to associate the
    ### Instruction: pattern with ### Response:
    """
    return (
        f"### Instruction:\n{pair['instruction']}\n\n"
        f"### Response:\n{pair['response']}\n\n"
    )


# Join all pairs into one string and repeat
# Repetition is necessary because our tiny model
# needs many token passes to learn the pattern.
# Real LLMs use thousands of unique pairs instead.
sft_text = "".join(format_sft(p) for p in sft_pairs) * 200
sft_data   = torch.tensor(encode(sft_text), dtype=torch.long)

<div class="alert alert-block alert-warning">What is happening here? How will you solve it? </div>

Here are some fixes:

    1. encode = lambda s: [stoi[c] for c in s if c in stoi]

    2. UNK = stoi.get(" ", 0)   # fallback to space, or index 0
       encode = lambda s: [stoi.get(c, UNK) for c in s]

    3. # Find all new characters in the SFT data
        new_chars = sorted(set(c for p in sft_pairs
                           for text in [p["instruction"], p["response"]]
                           for c in text
                           if c not in stoi))
    
       # Extend stoi and itos
        for ch in new_chars:
            idx = len(stoi)
            stoi[ch] = idx
            itos[idx] = ch
        
        vocab_size = len(stoi)   # ← update this too before rebuilding the model
        print(f"Added {len(new_chars)} new tokens: {new_chars}")
        print(f"New vocab size: {vocab_size}")

    4. # Keep old embeddings, add new random rows for new tokens
        old_embeddings = m.token_embedding_table.weight.data       # shape: [old_vocab, C]
        new_embeddings = torch.randn(len(new_chars), old_vocab_size) * 0.02
        
        m.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        m.token_embedding_table.weight.data[:old_vocab_size] = old_embeddings
        # new token rows are left as random init — they will learn during SFT

<div class="alert alert-block alert-warning">
    Look at the four options and explain what the differences are and the issues with the approaches. HINT: Option 4 is the real-world solution. Think why the first 3 options don't work.
</div>

In [ ]:
# trying option 1

encode = lambda s: [stoi[c] for c in s if c in stoi]

sft_data   = torch.tensor(encode(sft_text), dtype=torch.long)

In [ ]:
n = int(0.9 * len(sft_data))
train_data = sft_data[:n]
val_data   = sft_data[n:]
print(f"SFT train tokens: {len(train_data):,}")
print(f"SFT val   tokens: {len(val_data):,}")

In [ ]:
# ─────────────────────────────────────────
# 14. Batch helper
# ─────────────────────────────────────────
block_size = 32
batch_size = 16

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix   = torch.randint(len(data) - block_size, (batch_size,))
    x    = torch.stack([data[i : i + block_size]     for i in ix])
    y    = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x, y

In [ ]:
# ─────────────────────────────────────────
# 15. Loss estimation (train + val)
# Shows whether the model is learning or
# overfitting on the fine-tuning data
# ─────────────────────────────────────────
@torch.no_grad()
def estimate_loss(eval_iters=50):
    out = {}
    m.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

In [ ]:
# ─────────────────────────────────────────
# 16. SFT Training loop
# Key difference from pre-training:
#   - Lower learning rate (1e-4 vs 3e-4)
#     → gentle nudge, not a full rewrite
#   - Fewer steps
#     → small curated dataset, stop before overfit
#   - Same loss function (cross-entropy)
#     → next-token prediction still drives learning
# ─────────────────────────────────────────
optimizer = torch.optim.AdamW(
    m.parameters(),
    lr=1e-4          # ← lower LR: we refine, not retrain
)

sft_steps = 2000

print(f"\nStarting SFT for {sft_steps} steps...")
print(f"{'Step':>6}  {'Train Loss':>10}  {'Val Loss':>10}")
print("-" * 32)

for step in range(sft_steps):
    # Evaluate periodically
    if step % 500 == 0:
        losses = estimate_loss()
        print(f"{step:>6}  {losses['train']:>10.4f}  {losses['val']:>10.4f}")

    xb, yb = get_batch("train")
    _, loss = m(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Final evaluation
losses = estimate_loss()
print(f"{sft_steps:>6}  {losses['train']:>10.4f}  {losses['val']:>10.4f}")

In [ ]:
# ─────────────────────────────────────────
# 17. Generate from the SFT model
# Compare against the pre-trained output
# to see behavioural change
# ─────────────────────────────────────────
print("\n--- SFT model output ---")
context = torch.tensor([encode("To be")], dtype=torch.long)
print(decode(m.generate(context, max_new_tokens=200)[0].tolist()))

In [ ]:
# ─────────────────────────────────────────
# 18. Save the fine-tuned model
# This is the "instruction model" checkpoint
# In HuggingFace: model.save_pretrained("sft-model/")
# ─────────────────────────────────────────
import os
os.makedirs("sft-model", exist_ok=True)

torch.save(m.state_dict(), "sft-model/bigram_sft.pt")

# Config stays the same — same architecture, new weights
with open("sft-model/config.json", "w") as f:
    json.dump({**config, "training_stage": "sft"}, f, indent=2)

print("\nSFT model saved to sft-model/")